# photonviz — every chart, in one notebook

Run all cells and scroll. Each cell draws one chart type; the last section sweeps
every one of them programmatically and reports anything that fails, so you can
check the whole surface without reading each picture.

In Google Colab, run `from google.colab import output; output.enable_custom_widget_manager()` once first.

In [ ]:
import numpy as np
import photonviz as pv

print("photonviz", pv.__version__)

rng = np.random.default_rng(7)
DARK = {"theme": "dark"}

# Shared data, so every cell below is cheap and reproducible.
n = 600
x = np.linspace(0, 10, n)
y = np.sin(x)
y2 = np.cos(x)

# A scalar field on a regular grid, for heatmap / contour / streamplot.
gc = gr = 64
gx, gy = np.meshgrid(np.linspace(-3, 3, gc), np.linspace(-3, 3, gr))
field = (np.sin(gx) * np.cos(gy)).ravel()
extent = {"x": [-3, 3], "y": [-3, 3]}

# Scattered samples, for the tri* family.
tn = 600
tx = rng.uniform(-3, 3, tn)
ty = rng.uniform(-3, 3, tn)
tz = np.sin(tx) * np.cos(ty) * np.exp(-(tx**2 + ty**2) / 12)

# A price series, for the finance charts.
bars = 200
close = 100 * np.cumprod(1 + rng.normal(0.0004, 0.012, bars))
open_ = np.concatenate([[100.0], close[:-1]])
high = np.maximum(open_, close) * (1 + np.abs(rng.normal(0, 0.004, bars)))
low = np.minimum(open_, close) * (1 - np.abs(rng.normal(0, 0.004, bars)))
bx = np.arange(bars, dtype=float)

# Labels and scores, for the classification charts.
labels = (rng.random(800) < 0.4).astype(int)
scores = np.clip(labels * 0.35 + rng.normal(0.35, 0.2, 800), 0, 1)
y_true = rng.integers(0, 4, 400)
y_pred = np.where(rng.random(400) < 0.78, y_true, rng.integers(0, 4, 400))

## Basic marks

In [ ]:
pv.line(x, y, name="sin", plot={**DARK, "legend": True, "title": "line"})

In [ ]:
pv.Plot(**DARK, title="step").step(x[:40], y[:40])

In [ ]:
pv.scatter(x, y2, size=4, plot={**DARK, "title": "scatter"})

In [ ]:
# A bubble chart: per-point size and colour.
pv.scatter(x[:60], y[:60], sizes=np.abs(y[:60]) * 22 + 4,
           plot={**DARK, "title": "scatter — per-point sizes"})

In [ ]:
pv.bar(np.arange(8), rng.random(8) * 5, plot={**DARK, "title": "bar"})

In [ ]:
pv.area(x, np.abs(y), plot={**DARK, "title": "area"})

In [ ]:
pv.grouped_bars(np.arange(5), [{"y": rng.random(5) * 3, "name": "a"},
                               {"y": rng.random(5) * 3, "name": "b"}],
                plot={**DARK, "legend": True, "title": "grouped_bars"})

In [ ]:
pv.stacked_bars(np.arange(5), [{"y": rng.random(5) * 3, "name": "a"},
                               {"y": rng.random(5) * 3, "name": "b"}],
                plot={**DARK, "legend": True, "title": "stacked_bars"})

In [ ]:
pv.stacked_area(x, [{"y": np.abs(y), "name": "a"}, {"y": np.abs(y2), "name": "b"}],
                plot={**DARK, "legend": True, "title": "stacked_area"})

In [ ]:
pv.stem(np.arange(24), rng.random(24), plot={**DARK, "title": "stem"})

In [ ]:
pv.errorbar(x[:30], y[:30], yerr=np.full(30, 0.15), plot={**DARK, "title": "errorbar"})

In [ ]:
pv.errorbar(x, y, yerr=np.full(n, 0.2), band=True, plot={**DARK, "title": "errorbar — band"})

In [ ]:
pv.quiver(gx.ravel(), gy.ravel(), np.sin(gy).ravel(), np.cos(gx).ravel(),
          plot={**DARK, "title": "quiver"})

In [ ]:
pv.pie([5, 3, 2, 1], labels=["a", "b", "c", "d"], plot={**DARK, "equalAspect": True, "title": "pie"})

In [ ]:
pv.patches([{"x": [0, 1, 1, 0], "y": [0, 0, 1, 1], "color": "#60a5fa"},
            {"x": [1.2, 2.2, 1.7], "y": [0, 0, 1], "color": "#f472b6"}],
           plot={**DARK, "title": "patches"})

In [ ]:
pv.graph([[0, 1], [1, 2], [2, 3], [3, 0], [0, 2]], plot={**DARK, "title": "graph — force layout"})

In [ ]:
# Guide lines are annotations, not series.
(pv.Plot(**DARK, title="annotations")
   .line(x, y, name="sin")
   .hline(0, color="#64748b")
   .vline(np.pi, color="#f472b6")
   .annotate("band", dim="y", start=-0.5, end=0.5, color="rgba(96,165,250,0.15)"))

## Fields and rasters

matplotlib's gridded and vector-field types, under the same names.

In [ ]:
pv.heatmap(field, gc, gr, extent, colormap="magma", plot={**DARK, "title": "heatmap"})

In [ ]:
pv.contour(field, gc, gr, extent, levels=10, plot={**DARK, "title": "contour"})

In [ ]:
pv.contourf(field, gc, gr, extent, levels=12, lines=True, plot={**DARK, "title": "contourf"})

In [ ]:
# pcolormesh takes cell *edges*, so the cells need not be evenly spaced.
x_edges = np.concatenate([np.linspace(0, 5, 12), np.linspace(6, 30, 9)])
y_edges = np.geomspace(1, 200, 16)
cells = np.abs(np.sin(np.arange((len(x_edges) - 1) * (len(y_edges) - 1)) * 0.3)) * 4
pv.pcolormesh(cells, x_edges, y_edges, plot={**DARK, "title": "pcolormesh — uneven cells"})

In [ ]:
pv.hexbin(rng.normal(0, 1, 20_000), rng.normal(0, 1, 20_000),
          plot={**DARK, "title": "hexbin"})

In [ ]:
pv.hist2d(rng.normal(0, 1, 40_000), rng.normal(0, 1.6, 40_000), bins=[64, 44],
          colormap="magma", plot={**DARK, "title": "hist2d"})

In [ ]:
pv.eventplot([np.sort(rng.random(50 + i * 6) * 10) for i in range(8)],
             color="#a78bfa", plot={**DARK, "title": "eventplot"})

In [ ]:
# Streamlines of a dipole, each coloured by its own mean speed.
m = 48
sx, sy = np.meshgrid(np.linspace(-2, 2, m), np.linspace(-2, 2, m))
d1 = np.maximum(0.05, (sx + 1) ** 2 + sy ** 2)
d2 = np.maximum(0.05, (sx - 1) ** 2 + sy ** 2)
pv.streamplot(((sx + 1) / d1 - (sx - 1) / d2).ravel(), (sy / d1 - sy / d2).ravel(),
              m, m, {"x": [-2, 2], "y": [-2, 2]}, colormap="plasma", density=1.1,
              plot={**DARK, "title": "streamplot", "equalAspect": True})

In [ ]:
# Wind barbs: speed is read off the ticks, not the length.
bn = 9
wx, wy = np.meshgrid(np.arange(bn), np.arange(bn))
speed = np.linspace(2, 65, bn * bn)
angle = np.linspace(0, 2 * np.pi, bn * bn)
pv.barbs(wx.ravel(), wy.ravel(), speed * np.cos(angle), speed * np.sin(angle),
         plot={**DARK, "title": "barbs — 2 to 65 kt"})

In [ ]:
# A bitmap placed in data space.
PNG = ("data:image/png;base64,iVBORw0KGgoAAAANSUhEUgAAAAEAAAABCAYAAAAfFcSJAAAADUlEQVR42m"
       "P8z8BQDwAEhQGAhKmMIQAAAABJRU5ErkJggg==")
pv.Plot(**DARK, title="image").image(PNG, {"x": [0, 1], "y": [0, 1]})

## Scattered samples — the `tri*` family

Irregular points get a Delaunay triangulation first. Pass `triangles=` if the
connectivity is already part of your data.

In [ ]:
pv.triplot(tx, ty, showPoints=True, plot={**DARK, "title": "triplot"})

In [ ]:
pv.tripcolor(tx, ty, tz, edges=True, plot={**DARK, "title": "tripcolor"})

In [ ]:
pv.tricontour(tx, ty, tz, levels=10, plot={**DARK, "title": "tricontour"})

In [ ]:
pv.tricontourf(tx, ty, tz, levels=12, lines=True, plot={**DARK, "title": "tricontourf"})

## Distributions and statistics

In [ ]:
pv.histogram(rng.normal(0, 1, 20_000), bins=48, plot={**DARK, "title": "histogram"})

In [ ]:
groups = [{"x": i, "values": rng.normal(i * 0.4, 0.5 + i * 0.1, 400)} for i in range(5)]
pv.box(groups, plot={**DARK, "title": "box — Tukey quartiles"})

In [ ]:
pv.box(groups, violin=True, plot={**DARK, "title": "box + violin (KDE)"})

In [ ]:
pv.ecdf(rng.normal(0, 1, 4000), plot={**DARK, "title": "ecdf"})

In [ ]:
pv.regression(x, y + rng.normal(0, 0.2, n), band=2,
              plot={**DARK, "title": "regression — OLS + band"})

In [ ]:
pv.regression(x, y + rng.normal(0, 0.25, n), method="loess",
              plot={**DARK, "title": "regression — LOESS"})

In [ ]:
pv.corr_matrix([rng.normal(0, 1, 300) for _ in range(6)], names=list("abcdef"),
               plot={**DARK, "title": "corr_matrix"})

In [ ]:
sr = 500
sig = np.sin(2 * np.pi * 50 * np.arange(4096) / sr) + rng.normal(0, 0.4, 4096)
pv.psd(sig, sampleRate=sr, plot={**DARK, "title": "psd — Welch"})

## Finance

In [ ]:
pv.candlestick(bx, open_, high, low, close, plot={**DARK, "title": "candlestick"})

In [ ]:
pv.ohlc(bx, open_, high, low, close, plot={**DARK, "title": "ohlc"})

In [ ]:
pv.heikin_ashi(bx, open_, high, low, close, plot={**DARK, "title": "heikin_ashi"})

In [ ]:
pv.bollinger(bx, close, plot={**DARK, "title": "bollinger"})

In [ ]:
pv.renko(close, brick_size=2.0, plot={**DARK, "title": "renko"})

In [ ]:
pv.depth([[99 + i * 0.02, 50 - i] for i in range(50)],
         [[100 + i * 0.02, 1 + i] for i in range(50)],
         plot={**DARK, "title": "depth"})

In [ ]:
pv.volume_profile(close, rng.random(bars) * 1000,
                  plot={**DARK, "title": "volume_profile"})

In [ ]:
pv.drawdown(close, plot={**DARK, "title": "drawdown"})

## Machine learning

In [ ]:
pv.confusion_matrix(y_true, y_pred, classNames=list("abcd"),
                    plot={**DARK, "title": "confusion_matrix"})

In [ ]:
pv.roc_curve(scores, labels, plot={**DARK, "legend": True, "title": "roc_curve"})

In [ ]:
pv.pr_curve(scores, labels, plot={**DARK, "legend": True, "title": "pr_curve"})

In [ ]:
pv.calibration(scores, labels, plot={**DARK, "title": "calibration"})

In [ ]:
pv.lift_curve(scores, labels, plot={**DARK, "legend": True, "title": "lift_curve"})

In [ ]:
pv.embedding(rng.normal(0, 1, 600), rng.normal(0, 1, 600), labels=rng.integers(0, 4, 600),
             plot={**DARK, "title": "embedding"})

In [ ]:
pv.feature_importance(["age", "income", "tenure", "region"], [0.42, 0.31, 0.18, 0.09],
                      plot={**DARK, "title": "feature_importance"})

In [ ]:
pv.shap_beeswarm(["f0", "f1", "f2", "f3"],
                 [rng.normal(0, s, 300) for s in (1.0, 0.6, 0.35, 0.2)],
                 plot={**DARK, "title": "shap_beeswarm"})

In [ ]:
pv.training_curves([{"name": "train", "values": np.exp(-np.linspace(0, 3, 80)) + rng.normal(0, 0.02, 80)},
                    {"name": "val", "values": np.exp(-np.linspace(0, 2.6, 80)) + rng.normal(0, 0.04, 80)}],
                   plot={**DARK, "legend": True, "title": "training_curves"})

In [ ]:
pv.learning_curve(np.arange(10) * 100 + 100, np.linspace(0.60, 0.95, 10), np.linspace(0.55, 0.88, 10),
                  plot={**DARK, "legend": True, "title": "learning_curve"})

In [ ]:
pv.partial_dependence(np.linspace(0, 1, 60), np.sin(np.linspace(0, 3, 60)),
                      plot={**DARK, "title": "partial_dependence"})

In [ ]:
pv.attention_map(rng.random(144), queries=12, keys=12, plot={**DARK, "title": "attention_map"})

In [ ]:
pv.ridgeline([{"name": f"g{i}", "values": rng.normal(i * 0.5, 1, 400)} for i in range(6)],
             plot={**DARK, "title": "ridgeline"})

In [ ]:
pv.pred_vs_actual(y, y + rng.normal(0, 0.15, n), plot={**DARK, "title": "pred_vs_actual"})

In [ ]:
pv.residuals(y, y + rng.normal(0, 0.15, n), plot={**DARK, "title": "residuals"})

In [ ]:
# The classifier's prediction over a grid, with the training points on top.
side = 40
bxg, byg = np.meshgrid(np.linspace(-3, 3, side), np.linspace(-3, 3, side))
grid = ((bxg + byg) > 0).astype(int).ravel()
px_, py_ = rng.normal(0, 1, 300), rng.normal(0, 1, 300)
pv.decision_boundary(grid, side, side, {"x": [-3, 3], "y": [-3, 3]},
                     points={"x": px_, "y": py_, "labels": ((px_ + py_) > 0).astype(int)},
                     plot={**DARK, "title": "decision_boundary"})

## Diagrams

In [ ]:
pv.treemap([{"label": f"n{i}", "value": 10 + rng.random() * 90} for i in range(9)],
           plot={**DARK, "showToolbar": False, "title": "treemap"})

In [ ]:
pv.funnel([{"label": s, "value": v} for s, v in zip("ABCDE", [1000, 720, 480, 300, 140])],
          plot={**DARK, "showToolbar": False, "title": "funnel"})

In [ ]:
pv.sunburst({"label": "root", "children": [
    {"label": "a", "children": [{"label": "a1", "value": 5}, {"label": "a2", "value": 3}]},
    {"label": "b", "value": 6}]},
    plot={**DARK, "showToolbar": False, "title": "sunburst"})

In [ ]:
pv.gauge(72, min=0, max=100, plot={**DARK, "showToolbar": False, "title": "gauge"})

In [ ]:
pv.sankey(["source", "mid", "a", "b"],
          [{"source": 0, "target": 1, "value": 10},
           {"source": 1, "target": 2, "value": 6},
           {"source": 1, "target": 3, "value": 4}],
          plot={**DARK, "showToolbar": False, "title": "sankey"})

In [ ]:
pv.chord([[0, 5, 3], [4, 0, 2], [1, 6, 0]],
         plot={**DARK, "showToolbar": False, "equalAspect": True, "title": "chord"})

In [ ]:
pv.parallel_coordinates(list("abcd"), [list(r) for r in rng.normal(0, 1, (80, 4))],
                        plot={**DARK, "title": "parallel_coordinates"})

## 3D — drag to orbit, wheel to zoom

In [ ]:
pv.surface(field, gc, gr)

In [ ]:
pv.scatter3d(rng.normal(0, 1, 1500), rng.normal(0, 1, 1500), rng.normal(0, 1, 1500))

In [ ]:
pv.line3d(np.cos(x * 2), np.sin(x * 2), x)

In [ ]:
pv.bar3d(np.tile(np.arange(8), 8).astype(float),
         np.repeat(np.arange(8), 8).astype(float),
         rng.random(64))

In [ ]:
pv.boxes3d([{"x": i, "y": 0, "z": 0, "w": 0.8, "h": 1 + i, "d": 0.8, "color": "#60a5fa"}
            for i in range(5)])

In [ ]:
pv.quiver3d(*(rng.normal(0, 1, 300) for _ in range(6)))

In [ ]:
vol = np.exp(-((np.mgrid[0:28, 0:28, 0:28] - 14) ** 2).sum(0) / 50).ravel()
pv.isosurface(vol, [28, 28, 28], 0.4)

In [ ]:
pv.volume(vol, [28, 28, 28])

In [ ]:
pv.contour3d(field, gc, gr)

In [ ]:
# A label pinned to a point in data space; it tracks the camera.
pv.Plot3D().surface(field, gc, gr).label(0, 1, 0, "peak", color="#fbbf24")

## Polar

In [ ]:
theta = np.linspace(0, 2 * np.pi, 400)
pv.polar_line(theta, np.abs(np.sin(3 * theta)), plot=DARK)

In [ ]:
pv.polar_scatter(theta[::12], rng.random(len(theta[::12])), plot=DARK)

## figsize and subplots

`figsize` is matplotlib's `(width, height)` **in inches** at `dpi` (100 by default).

In [ ]:
pv.Plot(figsize=(9, 3), **DARK, title="figsize=(9, 3)").line(x, y)

In [ ]:
fig, axes = pv.subplots(2, 2, figsize=(11, 6), sharex=True, title="subplots(2, 2)", **DARK)
axes[0, 0].line(x, y, name="sin").title("Loss")
axes[0, 1].scatter(x, y2, size=3).title("Accuracy")
axes[1, 0].histogram(rng.normal(0, 1, 5000), bins=40)
axes[1, 1].bar(np.arange(6), rng.random(6))
fig

In [ ]:
# Panels can span cells and pick their own kind.
fig = pv.figure(figsize=(11, 6), rows=2, cols=2, **DARK)
fig.add_subplot(colspan=2).line(bx, close, name="price")
fig.add_subplot(row=1, col=0).ecdf(rng.normal(0, 1, 3000))
fig.add_subplot(row=1, col=1, kind="polar").line(theta, np.abs(np.sin(2 * theta)))
fig

## Model architecture

Hand over a PyTorch / Keras / scikit-learn / ONNX model and the export happens in
Python, the layout in the browser. Here is a hand-written one so the notebook needs
no ML framework installed.

In [ ]:
MODEL = pv.from_layers([
    {"id": "in",  "type": "Input",   "shape": [3, 224, 224]},
    {"id": "c1",  "type": "Conv2d",  "shape": [64, 112, 112], "params": 9_408},
    {"id": "b1",  "type": "BatchNorm2d", "shape": [64, 112, 112], "params": 128},
    {"id": "r1",  "type": "ReLU",    "shape": [64, 112, 112]},
    {"id": "c2",  "type": "Conv2d",  "shape": [128, 56, 56], "params": 73_728},
    {"id": "p1",  "type": "MaxPool2d", "shape": [128, 28, 28]},
    {"id": "fc",  "type": "Linear",  "shape": [1000], "params": 512_000},
], name="tiny-resnet")

pv.model_graph(MODEL, direction="horizontal", plot={**DARK, "title": "model_graph"})

In [ ]:
# The same model as cuboids sized from each layer's output tensor.
pv.model_graph_3d(MODEL, labels="full",
                  plot={"aspectMode": "data", "projection": "orthographic", "showAxes": False})

In [ ]:
# `slices` draws a tensor as the many planes it really is, rather than one block.
pv.model_graph_3d(MODEL, labels="short", slices="channels", maxSlices=24,
                  plot={"aspectMode": "data", "projection": "orthographic", "showAxes": False})

## Self-check — build every chart and report failures

This does not draw anything. It constructs every chart type above, encodes it the
way the widget would, and prints anything that raised. If this prints `all N ok`,
the whole Python surface is intact on your machine.

In [ ]:
from photonviz._arrays import encode_spec

def build_all():
    """One thunk per chart type, so a failure names the chart that failed."""
    P = lambda **kw: pv.Plot(**DARK, **kw)
    edges = [[0, 1], [1, 2], [2, 3]]
    x_edges = np.linspace(0, 10, 13)
    y_edges = np.geomspace(1, 100, 9)
    cells = rng.random(12 * 8)
    side = 24
    bxg, byg = np.meshgrid(np.linspace(-3, 3, side), np.linspace(-3, 3, side))
    grid = ((bxg + byg) > 0).astype(int).ravel()
    vol_s = np.exp(-((np.mgrid[0:16, 0:16, 0:16] - 8) ** 2).sum(0) / 30).ravel()
    th = np.linspace(0, 2 * np.pi, 200)
    return {
        # basic marks
        "line": lambda: P().line(x, y), "step": lambda: P().step(x, y),
        "scatter": lambda: P().scatter(x, y), "bar": lambda: P().bar(np.arange(6), rng.random(6)),
        "area": lambda: P().area(x, np.abs(y)),
        "grouped_bars": lambda: P().grouped_bars(np.arange(4), [{"y": rng.random(4)}]),
        "stacked_bars": lambda: P().stacked_bars(np.arange(4), [{"y": rng.random(4)}]),
        "stacked_area": lambda: P().stacked_area(x, [{"y": np.abs(y)}]),
        "stem": lambda: P().stem(np.arange(12), rng.random(12)),
        "errorbar": lambda: P().errorbar(x, y, yerr=np.full(n, 0.1)),
        "quiver": lambda: P().quiver(gx.ravel(), gy.ravel(), np.sin(gy).ravel(), np.cos(gx).ravel()),
        "pie": lambda: P().pie([3, 2, 1]),
        "patches": lambda: P().patches([{"x": [0, 1, 1], "y": [0, 0, 1]}]),
        "graph": lambda: P().graph(edges),
        "image": lambda: P().image("data:image/png;base64,iVBORw0KGgo=", {"x": [0, 1], "y": [0, 1]}),
        # fields
        "heatmap": lambda: P().heatmap(field, gc, gr, extent),
        "contour": lambda: P().contour(field, gc, gr, extent),
        "contourf": lambda: P().contourf(field, gc, gr, extent, levels=8),
        "pcolormesh": lambda: P().pcolormesh(cells, x_edges, y_edges),
        "hexbin": lambda: P().hexbin(rng.normal(0, 1, 2000), rng.normal(0, 1, 2000)),
        "hist2d": lambda: P().hist2d(rng.normal(0, 1, 2000), rng.normal(0, 1, 2000)),
        "eventplot": lambda: P().eventplot([np.sort(rng.random(20)) for _ in range(3)]),
        "streamplot": lambda: P().streamplot(np.sin(gy).ravel(), np.cos(gx).ravel(), gc, gr, extent),
        "barbs": lambda: P().barbs(gx.ravel()[:40], gy.ravel()[:40],
                                   np.sin(gy).ravel()[:40] * 30, np.cos(gx).ravel()[:40] * 30),
        # triangulation
        "triplot": lambda: P().triplot(tx, ty),
        "tripcolor": lambda: P().tripcolor(tx, ty, tz),
        "tricontour": lambda: P().tricontour(tx, ty, tz),
        "tricontourf": lambda: P().tricontourf(tx, ty, tz),
        # distributions / stats
        "histogram": lambda: P().histogram(rng.normal(0, 1, 2000)),
        "box": lambda: P().box([{"x": 0, "values": rng.normal(0, 1, 100)}]),
        "violin": lambda: P().box([{"x": 0, "values": rng.normal(0, 1, 100)}], violin=True),
        "ecdf": lambda: P().ecdf(rng.normal(0, 1, 1000)),
        "regression": lambda: P().regression(x, y, band=2),
        "corr_matrix": lambda: P().corr_matrix([rng.normal(0, 1, 100) for _ in range(3)]),
        "psd": lambda: P().psd(rng.normal(0, 1, 2048), sampleRate=500),
        # finance
        "candlestick": lambda: P().candlestick(bx, open_, high, low, close),
        "ohlc": lambda: P().ohlc(bx, open_, high, low, close),
        "heikin_ashi": lambda: P().heikin_ashi(bx, open_, high, low, close),
        "bollinger": lambda: P().bollinger(bx, close),
        "renko": lambda: P().renko(close, brick_size=2.0),
        "depth": lambda: P().depth([[99, 10]], [[101, 10]]),
        "volume_profile": lambda: P().volume_profile(close, rng.random(bars) * 100),
        "drawdown": lambda: P().drawdown(close),
        # machine learning
        "confusion_matrix": lambda: P().confusion_matrix(y_true, y_pred),
        "roc_curve": lambda: P().roc_curve(scores, labels),
        "pr_curve": lambda: P().pr_curve(scores, labels),
        "calibration": lambda: P().calibration(scores, labels),
        "lift_curve": lambda: P().lift_curve(scores, labels),
        "embedding": lambda: P().embedding(rng.normal(0, 1, 200), rng.normal(0, 1, 200)),
        "feature_importance": lambda: P().feature_importance(list("abc"), [3, 2, 1]),
        "shap_beeswarm": lambda: P().shap_beeswarm(list("abc"), [rng.normal(0, 1, 60) for _ in range(3)]),
        "training_curves": lambda: P().training_curves([{"name": "t", "values": rng.random(30)}]),
        "learning_curve": lambda: P().learning_curve(np.arange(5) * 10, rng.random(5), rng.random(5)),
        "partial_dependence": lambda: P().partial_dependence(np.linspace(0, 1, 20), rng.random(20)),
        "attention_map": lambda: P().attention_map(rng.random(64), queries=8, keys=8),
        "ridgeline": lambda: P().ridgeline([{"name": "g", "values": rng.normal(0, 1, 100)}]),
        "pred_vs_actual": lambda: P().pred_vs_actual(y, y + 0.1),
        "residuals": lambda: P().residuals(y, y + 0.1),
        "decision_boundary": lambda: P().decision_boundary(
            grid, side, side, {"x": [-3, 3], "y": [-3, 3]},
            points={"x": rng.normal(0, 1, 100), "y": rng.normal(0, 1, 100),
                    "labels": rng.integers(0, 2, 100)}),
        "model_graph": lambda: P().model_graph(MODEL),
        # diagrams
        "treemap": lambda: P().treemap([{"label": "a", "value": 1}]),
        "funnel": lambda: P().funnel([{"label": "a", "value": 9}, {"label": "b", "value": 4}]),
        "sunburst": lambda: P().sunburst({"label": "r", "children": [{"label": "a", "value": 1}]}),
        "gauge": lambda: P().gauge(50),
        "sankey": lambda: P().sankey(["a", "b"], [{"source": 0, "target": 1, "value": 1}]),
        "chord": lambda: P().chord([[0, 1], [1, 0]]),
        "parallel_coordinates": lambda: P().parallel_coordinates(list("ab"), [[0, 1], [1, 0]]),
        # 3D
        "surface": lambda: pv.Plot3D().surface(field, gc, gr),
        "scatter3d": lambda: pv.Plot3D().scatter3d(*(rng.normal(0, 1, 200) for _ in range(3))),
        "line3d": lambda: pv.Plot3D().line3d(np.cos(x), np.sin(x), x),
        "bar3d": lambda: pv.Plot3D().bar3d(np.tile(np.arange(4), 4).astype(float),
                                           np.repeat(np.arange(4), 4).astype(float), rng.random(16)),
        "boxes3d": lambda: pv.Plot3D().boxes3d([{"x": 0, "y": 0, "z": 0, "w": 1, "h": 1, "d": 1}]),
        "quiver3d": lambda: pv.Plot3D().quiver3d(*(rng.normal(0, 1, 60) for _ in range(6))),
        "isosurface": lambda: pv.Plot3D().isosurface(vol_s, [16, 16, 16], 0.4),
        "volume": lambda: pv.Plot3D().volume(vol_s, [16, 16, 16]),
        "contour3d": lambda: pv.Plot3D().contour3d(field, gc, gr),
        "model_graph_3d": lambda: pv.Plot3D().model_graph(MODEL),
        "label3d": lambda: pv.Plot3D().surface(field, gc, gr).label(0, 0, 0, "peak"),
        # polar
        "polar_line": lambda: pv.Polar(**DARK).line(th, np.abs(np.sin(3 * th))),
        "polar_scatter": lambda: pv.Polar(**DARK).scatter(th[::10], rng.random(20)),
        # figures
        "figsize": lambda: pv.Plot(figsize=(8, 3), **DARK).line(x, y),
        "subplots": lambda: pv.subplots(2, 2, figsize=(8, 4), **DARK)[0],
        "figure_spans": lambda: pv.figure(figsize=(8, 4), rows=2, cols=2, **DARK)
                                  .add_subplot(colspan=2).line(x, y)._parent,
    }

failed = {}
built = build_all()
for name, make in built.items():
    try:
        chart = make()
        encode_spec(chart._raw_spec())          # the same encoding the widget receives
    except Exception as exc:
        failed[name] = f"{type(exc).__name__}: {exc}"

if failed:
    print(f"{len(failed)} of {len(built)} FAILED:")
    for k, v in failed.items():
        print(f"  {k}: {v}")
else:
    print(f"all {len(built)} ok")

---

**Something not drawing?** Every keyword here maps 1:1 onto the TypeScript options,
so the [chart reference](https://coredumpdev.github.io/photon/docs/charts/2d) applies
verbatim. `chart.to_spec()` shows exactly what was sent to the browser.